<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 5 · Martes — Regresión Lineal y Métricas</h1>
<h3>Nuestro primer modelo de ML real</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final de la clase vas a poder:

1. Entender **qué es una regresión lineal** y cuándo usarla.
2. Entrenar tu **primer modelo** con `LinearRegression` de sklearn.
3. Evaluar el modelo con las **3 métricas principales**: MAE, MSE/RMSE, R².
4. **Interpretar** lo que dice cada métrica (qué tan bueno es el modelo).
5. Resolver **4 ejercicios prácticos** que veremos en vivo.

> Requisito previo: la clase del lunes (Estandarización + Train/Test).

# 1. ¿Qué es la regresión lineal?

La **regresión lineal** es el modelo más simple de Machine Learning. Predice un **número continuo** (precio, edad, ventas, temperatura) a partir de otras variables.

### 🧠 La idea con un ejemplo

Imagina que tienes pares de datos (horas estudiadas → nota obtenida):

| Horas | Nota |
|---|---|
| 1 | 3.0 |
| 2 | 4.0 |
| 3 | 5.0 |
| 4 | 6.0 |

La regresión lineal busca **la línea recta que mejor pasa por estos puntos**, con la forma:

$$ \hat{y} = a \cdot x + b $$

Donde:
- $a$ = **pendiente** (cuánto sube `y` por cada unidad de `x`)
- $b$ = **intercepto** (valor de `y` cuando `x` es 0)

En este ejemplo, la línea sería `nota = 1 × horas + 2`. Si estudio 5 horas, predigo nota 7.

### Con varias variables

$$ \hat{y} = a_1 x_1 + a_2 x_2 + a_3 x_3 + \dots + b $$

El modelo aprende un **peso** ($a_i$) para cada variable, y los suma para predecir.

### 👀 Visualización del concepto

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Datos simulados (horas estudiadas vs nota)
horas = np.array([1, 2, 3, 4, 5, 6, 7])
nota  = np.array([3.0, 4.1, 4.9, 5.8, 7.2, 7.8, 8.5])

# La regresión lineal busca la mejor línea recta que pasa por los puntos
a, b = np.polyfit(horas, nota, 1)
x_linea = np.linspace(0, 8, 100)
y_linea = a * x_linea + b

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(horas, nota, s=80, color='steelblue', zorder=3, label='Datos reales')
ax.plot(x_linea, y_linea, color='red', linewidth=2.5, label=f'Línea: y = {a:.2f}·x + {b:.2f}')
ax.set_xlabel('Horas estudiadas')
ax.set_ylabel('Nota')
ax.set_title('Regresión lineal — la mejor línea recta')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print(f'👉 Si alguien estudia 5.5 horas, predecimos nota: {a*5.5 + b:.2f}')

# 2. Primer modelo con Scikit-Learn

Vamos a entrenar nuestro primer modelo **de verdad** prediciendo el precio de casas en California.

**Flujo completo (5 pasos):**

1. Cargar datos
2. Separar features (X) y target (y)
3. Dividir en train/test
4. Crear el modelo y entrenarlo (`.fit()`)
5. Predecir y evaluar (`.predict()`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# 1. Cargar
df = pd.read_csv('data/housing.csv').drop(columns=['ocean_proximity']).dropna()

# 2. Separar features y target
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Estandarizamos (lo aprendimos ayer)
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc  = scaler.transform(X_test)

# 4. Crear modelo y entrenar
modelo = LinearRegression()
modelo.fit(X_train_esc, y_train)

print('✅ Modelo entrenado')
print(f'Pendientes aprendidas: {modelo.coef_.round(2)}')
print(f'Intercepto:            {modelo.intercept_:,.2f}')

In [ ]:
# 5. Predecir con el set de test
y_pred = modelo.predict(X_test_esc)

# Comparar real vs predicho (primeras 10)
comparacion = pd.DataFrame({
    'Real':     y_test.values[:10].round(0),
    'Predicho': y_pred[:10].round(0),
    'Error':    (y_test.values[:10] - y_pred[:10]).round(0)
})
comparacion

# 3. ¿Qué tan bueno es nuestro modelo? — Métricas

Tenemos predicciones — pero ¿cómo sabemos si son buenas? Para eso usamos **métricas**.

### Las 3 métricas que TIENES que conocer

| Métrica | Fórmula simple | ¿Qué mide? | ¿Cuándo se prefiere? |
|---|---|---|---|
| **MAE** (Mean Absolute Error) | promedio de \|real - predicho\| | Error promedio en las mismas unidades que `y` | Cuando todos los errores cuentan igual |
| **MSE** (Mean Squared Error) | promedio de (real - predicho)² | Error promedio elevado al cuadrado | Cuando errores grandes son MUY malos |
| **RMSE** (Root MSE) | √MSE | Como MSE pero en las unidades originales | El más usado en regresión |
| **R²** (R cuadrado) | varianza explicada | % de la varianza que el modelo captura | Para comparar modelos entre sí |

### Cómo leer R²

| R² | Significa |
|---|---|
| 1.0 | Modelo perfecto (nunca pasa en la vida real) |
| 0.7 - 0.9 | Muy buen modelo |
| 0.4 - 0.7 | Modelo decente |
| 0.0 | Predice igual que tirar el promedio |
| Negativo | El modelo es PEOR que tirar el promedio (mal) |

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('📊 Evaluación del modelo')
print('─' * 50)
print(f'MAE:   {mae:>12,.0f}    (error promedio en dólares)')
print(f'MSE:   {mse:>12,.0f}    (en dólares cuadrados — difícil de leer)')
print(f'RMSE:  {rmse:>12,.0f}    (raíz del MSE → en dólares)')
print(f'R²:    {r2:>12.4f}    ({r2*100:.1f}% de la varianza explicada)')

### 👀 Visualización del rendimiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Real vs Predicho — si el modelo fuera perfecto los puntos estarían en la diagonal
axes[0].scatter(y_test, y_pred, alpha=0.3, s=15, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             color='red', linewidth=2, linestyle='--', label='Predicción perfecta')
axes[0].set_xlabel('Precio real')
axes[0].set_ylabel('Precio predicho')
axes[0].set_title(f'Real vs Predicho (R² = {r2:.3f})')
axes[0].legend()

# Distribución de errores (residuos)
residuos = y_test - y_pred
axes[1].hist(residuos, bins=50, color='#C0504D', edgecolor='white')
axes[1].axvline(0, color='black', linewidth=2, linestyle='--')
axes[1].set_title('Distribución de errores (residuos)')
axes[1].set_xlabel('Error (real - predicho)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

print('👉 Idealmente los residuos están centrados en 0 y tienen forma de campana.')

### 🪄 Atajo: `modelo.score()`

En sklearn, `modelo.score(X_test, y_test)` devuelve directamente el **R²**. Es el atajo más usado.

In [ ]:
r2_atajo = modelo.score(X_test_esc, y_test)
print(f'R² con el atajo: {r2_atajo:.4f}')
print(f'R² calculado:    {r2:.4f}')
print('Son idénticos ✅')

---
# 🏋️ Ejercicios prácticos

**Datasets:**
- `data/housing.csv` — precios de casas (lo trabajamos hoy)
- `data/life_expectancy.csv` — esperanza de vida

> Para cada ejercicio: intenta primero, después lo revisamos juntos.

## Ejercicio 1 — Regresión simple con UNA sola variable

**Tarea:**
1. Cargar `data/housing.csv`.
2. Usar **solo la columna `median_income`** como feature (X).
3. Target: `median_house_value`.
4. Train/test split (80/20, random_state=42).
5. Entrenar `LinearRegression`.
6. Reportar el R² en el test.

**Pista:** cuando X es una sola columna, asegúrate de pasarla como DataFrame (doble corchete `df[['columna']]`).

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df = pd.read_csv('data/housing.csv')
X = df[['median_income']]
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print(f'R²: {modelo.score(X_test, y_test):.4f}')
print(f'Pendiente: {modelo.coef_[0]:,.2f}')
print(f'Intercepto: {modelo.intercept_:,.2f}')
# El ingreso explica ~47% de la varianza del precio — buen punto de partida con una sola variable.
```
</details>

## Ejercicio 2 — Comparar las 3 métricas

**Tarea:** Usando el modelo del Ejercicio 1:

1. Calcular **MAE, RMSE y R²**.
2. Imprimir cada uno con un mensaje claro.
3. Responder en un comentario: si la mediana del precio real es $180.000, ¿el RMSE es aceptable?


In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = modelo.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'MAE:  {mae:,.0f}')
print(f'RMSE: {rmse:,.0f}')
print(f'R²:   {r2:.4f}')

# Si la mediana es $180.000 y el RMSE es ~80.000, el error es del 44% del precio típico.
# Es alto — con una sola variable es esperable; mañana lo mejoramos con más features.
```
</details>

## Ejercicio 3 — Multivariable: ¿mejora?

**Tarea:**
1. Usar el flujo completo con TODAS las columnas numéricas (quitando `ocean_proximity`).
2. Estandarizar correctamente (fit solo con train).
3. Entrenar `LinearRegression`.
4. Comparar el R² con el del Ejercicio 1. ¿Mejoró?

**Pista:** usa `.dropna()` para los nulos por ahora.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

df = pd.read_csv('data/housing.csv').drop(columns=['ocean_proximity']).dropna()
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc  = scaler.transform(X_test)

modelo = LinearRegression()
modelo.fit(X_train_esc, y_train)

print(f'R² con TODAS las features: {modelo.score(X_test_esc, y_test):.4f}')
# Suele subir a ~0.62 — mucho mejor que solo con income (~0.47).
```
</details>

## Ejercicio 4 — Aplicar todo al dataset de esperanza de vida

**Tarea:**
1. Cargar `data/life_expectancy.csv` con `index_col='CountryYear'`.
2. Quitar las columnas `Life expectancy` (target) y `Status`. Aplicar `.dropna()`.
3. Definir X (todas las restantes) y `y = Life expectancy`.
4. Train/test split 80/20, random_state=7.
5. Estandarizar correctamente.
6. Entrenar `LinearRegression`.
7. Reportar MAE, RMSE y R². ¿Es un buen modelo?
8. **Bonus:** ¿Cuál es la variable con mayor coeficiente (en valor absoluto)? Eso indica cuál es la más influyente.

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv('data/life_expectancy.csv', index_col='CountryYear').dropna()
X = df.drop(columns=['Life expectancy', 'Status'])
y = df['Life expectancy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)

scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)
X_test_esc  = scaler.transform(X_test)

modelo = LinearRegression()
modelo.fit(X_train_esc, y_train)
y_pred = modelo.predict(X_test_esc)

print(f'MAE:  {mean_absolute_error(y_test, y_pred):.2f} años')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} años')
print(f'R²:   {modelo.score(X_test_esc, y_test):.4f}')

# Bonus — variable más influyente
importancia = pd.Series(np.abs(modelo.coef_), index=X.columns).sort_values(ascending=False)
print('\nTop 5 variables más influyentes:')
print(importancia.head())
```
</details>

---
## 📌 Cierre del día

Hoy:

- ✅ Aprendiste qué es la **regresión lineal** y la idea de la "mejor recta"
- ✅ Entrenaste tu **primer modelo real** con `LinearRegression`
- ✅ Calculaste **MAE, MSE, RMSE y R²** y aprendiste a leerlos
- ✅ Visualizaste **real vs predicho** y la distribución de errores
- ✅ Vimos el atajo `modelo.score()` para R²

### 🔜 Próximas clases

- **Variables categóricas** (encoding con One Hot Encoder)
- **Pipelines** para automatizar todo el flujo
- **Otros modelos**: KNN, Árboles de Decisión, Random Forest
- **Optimización de hiperparámetros**

Nos vemos 🚀